# Lab 22: Evaluating grounded generation, and unit economics

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


In [2]:
assert Path("outputs/qa/dev.jsonl").exists(), "Run notebook 18 first: it writes outputs/qa/dev.jsonl."

In [3]:
# From the slide: The numeric-claim linter
import math, re
NUM = re.compile(r"""(?P<neg>\()?\$?\s*
  (?P<num>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)
  \s*(?P<unit>%|x\b|bn\b|mm\b|m\b|million\b|k\b)?
  (?(neg)\))""", re.X | re.I)
YEAR = re.compile(r"^(19|20)\d{2}$")
SCALE = {"bn": 1e9, "mm": 1e6, "m": 1e6, "million": 1e6,
         "k": 1e3}

def numbers_in(text):
    for m in NUM.finditer(text):
        raw, unit = m["num"], (m["unit"] or "").lower()
        a, b = m.start("num"), m.end("num")
        if (YEAR.match(raw) and not unit
                or text[a-1:a].isalpha()   # Q2, FY24
                or re.match(r"-[A-Za-z]", text[b:b+2])
                or re.search(r"(item|note)\s*$",
                             text[max(0, a-6):a], re.I)):
            continue               # 10-K, 8-K, Item 7
        v = float(raw.replace(",", ""))
        v *= SCALE.get(unit, 1)
        yield -v if m["neg"] else v

def lint(text, allowed, tol=0.005):     # [] == pass
    ok = lambda v: any(math.isclose(v, a, rel_tol=tol)
                       for a in allowed)
    return [v for v in numbers_in(text) if not ok(v)]

In [4]:
answers = [json.loads(l) for l in open("outputs/qa/dev.jsonl")]
text = {(r["doc_id"], r["block_id"]): r["text"] for r in sd.records("deal_a")}   # block ids repeat across documents
cells = {f["citation"]["cell"]: f["value"] for f in sd.seller_facts() if "cell" in f["citation"]}
def supports(cit, value):
    if cit["block_id"] in cells: return abs(cells[cit["block_id"]] - value) <= 0.005 * abs(value)
    return any(abs(v - value) <= 0.005 * abs(value) or abs(v - value * 100) <= 0.005 * abs(value * 100) for v in numbers_in(text.get((cit["doc_id"], cit["block_id"]), "")))
cited = supported = claims_with_support = n_claims = 0
for a in answers:
    for c in a["answer"]["claims"]:
        n_claims += 1; ok = False
        for cit in c["citations"]:
            cited += 1; s = all(supports(cit, n["value"]) for n in c["numbers"]); supported += s; ok |= s
        claims_with_support += ok
print(f"citation precision {supported}/{cited} | citation recall {claims_with_support}/{n_claims}")

citation precision 3/3 | citation recall 3/3


In [5]:
import csv
from sklearn.metrics import cohen_kappa_score
items = list(csv.DictReader(open("data/eval/judge_items.csv")))
judge = {l.item_id: l.supported for l in llm.structured(schemas.JudgeLabels, prompts.judge_messages([{"item_id": r["item_id"], "claim": r["claim"], "cited_text": r["cited_text"]} for r in items]), llm.P["llm"]["models"]["judge"]).labels}
a = [int(r["human_a"]) for r in items]; b = [int(r["human_b"]) for r in items]; j = [int(judge[r["item_id"]]) for r in items]
print(f"humans agree: kappa {cohen_kappa_score(a, b):.2f} | judge vs human A: kappa {cohen_kappa_score(a, j):.2f}")

humans agree: kappa 0.78 | judge vs human A: kappa 0.89


In [6]:
# Unit economics for one cited answer, from the price table and the human baseline (x0.5, x1, x2)
import pandas as pd
hm = pd.read_csv("data/reference/human_minutes.csv").set_index("task_category").loc["cited Q&A answer"]
msgs = prompts.answer_messages(sd.dev_tasks()["dev-w-01"]["turns"][0]["content"], sd.seller_facts(), [], sd.evidence_for("dev-w-01"))
tin = sum(len(m["content"]) for m in msgs) // 4
cost = llm.usd(llm.P["llm"]["models"]["large"], tin, 300)
for mult in (0.5, 1, 2):
    saved = hm["minutes_typical"] * mult
    print(f"minutes saved x{mult}: {saved:.0f} min of a $150/h associate = ${saved / 60 * 150:.2f} vs model cost ${cost:.4f}")
Path("reports").mkdir(exist_ok=True)
json.dump({"cs2_sample": {"cost_per_cited_answer_usd": round(cost, 5), "tokens_in_est": tin, "human_minutes_typical": int(hm["minutes_typical"])}}, open("reports/unit_economics_sample.json", "w"), indent=1)

minutes saved x0.5: 6 min of a $150/h associate = $15.00 vs model cost $0.0007
minutes saved x1: 12 min of a $150/h associate = $30.00 vs model cost $0.0007
minutes saved x2: 24 min of a $150/h associate = $60.00 vs model cost $0.0007


**Replay proof.** In your CS2 repository, rehearse offline replay with `LANTERN_LLM_MODE=replay dvc repro` and compare metrics across runs. This classroom notebook uses recorded model responses; its exercise report should be identical on a repeat run.

**Exercise 22 — where the humans disagree.** List every disagreement with the judge's label, distinguish rubric ambiguity from an incorrect label, and decide how to evaluate the judge. The supplied sample has **two** disagreements, not one. These are the kit's existing human labels, not new annotations collected from teammates.


In [7]:
labels = pd.DataFrame([
    {"item_id": r["item_id"], "claim": r["claim"], "cited_text": r["cited_text"],
     "human_a": x, "human_b": y, "judge": z}
    for r, x, y, z in zip(items, a, b, j)
])
assert len(labels) == len(items) == len(a) == len(b) == len(j) == 20
disagreements = labels[labels["human_a"] != labels["human_b"]].copy()
assert set(disagreements["item_id"]) == {"j16", "j20"}
print("All human disagreements, including the judge's label:")
display(disagreements)

agreed = labels[labels["human_a"] == labels["human_b"]]
assert len(agreed) == 18
comparisons = pd.DataFrame([
    {"comparison": "human A vs human B", "items": len(a), "kappa": cohen_kappa_score(a, b)},
    {"comparison": "judge vs human A", "items": len(a), "kappa": cohen_kappa_score(a, j)},
    {"comparison": "judge vs human B", "items": len(b), "kappa": cohen_kappa_score(b, j)},
    {"comparison": "judge vs agreed-label subset", "items": len(agreed),
     "kappa": cohen_kappa_score(agreed["human_a"], agreed["judge"])},
])
display(comparisons.round(3))

# Explain j16 using the cited adjacent revenue values.
growth_pct = (41.2 / 33.07 - 1) * 100
print(f"j16 derived revenue growth: {growth_pct:.3f}% (rounds to {round(growth_pct)}%).")
assert round(growth_pct) == 25
assert labels.set_index("item_id").loc["j20", "judge"] == 1
print("j20: the judge marked 24 months supported, but the citation says 36 months.")

report = {
    "label_source": "Provided classroom fixture; not new team annotations",
    "total_items": len(labels), "agreed_items": len(agreed),
    "disagreements": disagreements.to_dict(orient="records"),
    "comparisons": comparisons.to_dict(orient="records"),
    "j16_derived_growth_pct": growth_pct,
    "analysis": {
        "j16": "Rubric ambiguity: decide explicitly whether rounded derived values count as supported.",
        "j20": "Incorrect support label: 24 months contradicts the cited 36 months; human B and judge are wrong.",
    },
    "recommendation": "Report both individual comparisons and the agreed subset; adjudicate disagreements under a written rubric before full-set scoring. Do not treat exclusion as resolution.",
}
Path("reports/exercise22_judge_agreement.json").write_text(json.dumps(report, indent=2) + "\n")


All human disagreements, including the judge's label:


,item_id,claim,cited_text,human_a,human_b,judge
15,j16,Revenue grew about 25% in FY2025.,Revenue | 26.4 | 33.07 | 41.2,1,0,1
19,j20,The contract term is 24 months.,Term: 36 months,0,1,1


,comparison,items,kappa
0,human A vs human B,20,0.780
1,judge vs human A,20,0.886
2,judge vs human B,20,0.886
3,judge vs agreed-label subset,18,1.000


j16 derived revenue growth: 24.584% (rounds to 25%).
j20: the judge marked 24 months supported, but the citation says 36 months.


1445

**j16 — rubric ambiguity.** The cited revenue values imply about 24.584% growth, which rounds to 25%. Human A and the judge accept “about 25%”; human B rejects it. A proposed convention is to allow reproducible arithmetic from cited inputs, including “about” followed by a whole-percent rounding, provided the input periods and formula are explicit. Under that convention this claim is supported. If the rubric requires the percentage to appear verbatim, it is not. Settle and document the convention before scoring further items; do not change an annotator's original label silently.

**j20 — an error, not rounding ambiguity.** “24 months” contradicts “Term: 36 months.” Human A correctly rejects support. Human B and the judge incorrectly accept it. Their agreement does not make that answer correct.

**Which reference should score the judge?** Report agreement against both A and B rather than choosing whichever produces a better number. The 18-item subset where humans agree is a useful provisional reference, and the judge's kappa there is 1.0. However, that result excludes both disputed cases and hides the judge's j20 error. For final evaluation, adjudicate the two disagreements under a documented rubric and score the full set against the adjudicated labels. Agreement by exclusion is not a completed adjudication.

**How many human labels would justify trusting the judge on 500 memos?** There is no defensible universal count. This set of 20 binary examples is a pilot, not validation of 500 complete memos. Use independent double-labelling across memo sections and failure types, keep a held-out evaluation set, report uncertainty and consequential errors, and expand the sample until the predeclared error tolerance is supported. For the separate tutorial's 1–5 memo rubric, use quadratic-weighted kappa and Spearman correlation; do not present these binary fixture labels as new teammate annotations. Human review remains necessary for unresolved disagreements.
